## Pruebas finales del código de `getting_data.ipynb`

### 1. Obtención de los catálogos

In [ ]:
# ======================================================================================
# OBTENCIÓN DE DATOS (VERSIÓN FINAL Y CORREGIDA)
# ======================================================================================

import os
import json
import csv
import time

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


# ======================================================================================
# DRIVER
# ======================================================================================

def crear_driver():
    """Crea y devuelve una instancia de Chrome en modo headless."""

    opciones = Options()
    opciones.add_argument("--headless")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(options=opciones)


# Variable global del driver, se inicializa en control_flujo()
driver = None


# ======================================================================================
# BÚSQUEDA DE CATÁLOGO: EDITORIAL NORMAL (por páginas)
# ======================================================================================

def buscar_editorial(id_editorial, nombre_editorial, pag_inicio, pag_fin):
    """
    Recorre el catálogo de una editorial normal (hasta 200 páginas) iterando de pag_inicio a pag_fin.
    Devuelve los diccionarios con título, autor, precio y URL de cada libro.

    Parámetros:
    * **id_editorial:** el id asignado a la editorial
    * **nombre_editorial:** el nombre de la editorial
    * **pag_inicio:** página del catálogo web donde empezar a hacer scraping
    * **pag_final:** última página del catálogo donde hacer scraping

    Outputs: 
    * Lista de diccionarios con los resultados del scraping
    """
    libros = []
    num_pagina = pag_inicio

    while num_pagina <= pag_fin:
        url = f"https://www.todostuslibros.com/editoriales/{id_editorial}/catalogo?page={num_pagina}"

        # Mostrar progreso solo hasta la página 10 para no llenar la consola
        if num_pagina < 10:
            print(f"Página {num_pagina}...")
        elif num_pagina == 10:
            print("Página 10 y más...")

        driver.get(url)
        time.sleep(1)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        elementos = soup.select("h2 a")

        # Si no hay libros en la página, se acabó el catálogo
        if not elementos:
            print("Sin más resultados.")
            break

        for h2 in soup.select("h2"):
            a = h2.find("a")
            if not a:
                continue

            titulo = a.get_text(strip=True)

            # El autor está en el h3 inmediatamente después del h2
            h3 = h2.find_next_sibling("h3")
            autor = h3.get_text(strip=True) if h3 else ""

            # El precio está en el primer <strong> después del h2
            etiqueta_precio = h2.find_next("strong")
            precio = etiqueta_precio.get_text(strip=True) if etiqueta_precio else ""

            url_libro = a["href"] if a.get("href") else ""

            libros.append({
                "editorial": nombre_editorial,
                "titulo": titulo,
                "autor": autor,
                "precio": precio,
                "url": url_libro,
            })

        num_pagina += 1
        time.sleep(0.5)

    return libros


# ======================================================================================
# BÚSQUEDA DE CATÁLOGO: EDITORIAL GRANDE (por años, para superar el límite de 200 págs)
# ======================================================================================

def buscar_editorial_grande(id_editorial, nombre_editorial, anio_inicio, anio_fin):
    """
    Recorre el catálogo de una editorial grande filtrando por año, lo que permite superar el límite de 200 páginas por búsqueda.
    Para cada año itera todas las páginas disponibles hasta que no haya resultados.
    Devuelve los diccionarios con título, autor, precio y URL de cada libro.

    Parámetros:
    * **id_editorial:** el id asignado a la editorial
    * **nombre_editorial:** el nombre de la editorial
    * **anio_inicio:** página (del año) del catálogo web donde empezar a hacer scraping
    * **anio_final:** última página (del año) del catálogo donde hacer scraping

    Outputs: 
    * Lista de diccionario con los resultados del scraping
    """
    libros = []

    for anio in range(anio_inicio, anio_fin + 1):
        print(f"Año {anio}...")
        num_pagina = 1

        while True:
            url = f"https://www.todostuslibros.com/editoriales/{id_editorial}/catalogo?anios={anio}&page={num_pagina}"
            driver.get(url)
            time.sleep(1)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            elementos = soup.select("h2 a")
            
            # Si no hay libros, este año ya no tiene más páginas
            if not elementos:
                print(f"Sin más resultados en {anio}, página {num_pagina}.")
                break

            for h2 in soup.select("h2"):
                a = h2.find("a")
                if not a:
                    continue

                titulo = a.get_text(strip=True)

                h3 = h2.find_next_sibling("h3")
                autor = h3.get_text(strip=True) if h3 else ""

                etiqueta_precio = h2.find_next("strong")
                precio = etiqueta_precio.get_text(strip=True) if etiqueta_precio else ""

                url_libro = a["href"] if a.get("href") else ""

                libros.append({
                    "editorial": nombre_editorial,
                    "titulo": titulo,
                    "autor": autor,
                    "precio": precio,
                    "url": url_libro,
                })

            num_pagina += 1
            time.sleep(0.5)

    return libros


# ======================================================================================
# EXTRACCIÓN DE FICHA TÉCNICA Y SINOPSIS
# ======================================================================================

def extraer_datos(url):
    """
    Accede a la ficha de un libro y extrae todos los datos técnicos (ISBN, páginas, formato, etc.) y la sinopsis completa.
    Devuelve un diccionario con todos los campos encontrados.

    Parámetros:
    * **url:** link de la página de la ficha técnica

    Outputs: 
    * Diccionario con los resultados del scraping
    """
    driver.get(url)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # La ficha técnica está dentro de elementos <dl class="datos-tecnicos">
    secciones = soup.find_all("dl", class_="datos-tecnicos")

    nombres = []  # nombres de los campos (dt)
    datos = []    # valores de los campos (dd)

    for seccion in secciones:
        etiquetas_nombre = seccion.find_all("dt")
        etiquetas_dato = seccion.find_all("dd")

        # Extraer nombres de los campos técnicos
        for nombre in etiquetas_nombre:
            nombres.append(nombre.get_text(strip=True).replace(":", "").strip())

        # Extraer valores de los campos técnicos
        # Hay tres posibles estructuras dentro de cada <dd>:
        for dato in etiquetas_dato:

            # Caso 1: el valor está en uno o varios <a> (ej: categorías, editorial)
            enlaces = dato.find_all("a")
            if enlaces:
                lista_valores = [enlace.get_text(strip=True) for enlace in enlaces]
                # Si hay un solo valor lo guardamos como string, si hay varios como lista
                datos.append(lista_valores[0] if len(lista_valores) == 1 else lista_valores)
                continue

            # Caso 2: el valor está en un <span> (ej: idioma)
            span = dato.find("span")
            if span:
                datos.append(span.get_text(strip=True))
                continue

            # Caso 3: el valor está directamente en el <dd> (ej: dimensiones, páginas)
            # Usamos split/join para limpiar espacios y saltos de línea extra
            texto = dato.get_text()
            datos.append(" ".join(texto.split()))

    # Sinopsis completa: está en un div separado fuera de la ficha técnica
    sinopsis = soup.find("div", id="collapseSynopsis")
    nombres.append("Sinopsis")
    if sinopsis:
        # Extraemos párrafo a párrafo y los unimos con " | "
        parrafos = [p.get_text(strip=True) for p in sinopsis.find_all("p")]
        datos.append(" | ".join(parrafos))
    else:
        datos.append("Sin sinopsis")

    # Construir diccionario emparejando cada nombre con su dato
    ficha_tecnica = {}
    for nombre, dato in zip(nombres, datos):
        ficha_tecnica[nombre] = dato

    return ficha_tecnica


# ======================================================================================
# SCRAPING COMPLETO DE UNA EDITORIAL (búsqueda + fichas + guardado CSV)
# ======================================================================================

def scrapear_editorial(id_editorial, nombre_editorial, inicio, fin, es_grande):
    """
    Orquesta el proceso completo para una editorial:
    1. Busca todos los libros del catálogo en el intervalo indicado.
    2. Entra en cada ficha técnica y extrae los datos.
    3. Guarda el resultado en un CSV en la carpeta 'data/'.

    Parámetros:
    * **id_editorial:** identificador de la URL (ej: "debolsillo_179709")
    * **nombre_editorial:** nombre legible (ej: "DEBOLSILLO")
    * **inicio:** página o año de inicio
    * **fin:** página o año de fin
    * **es_grande:** True si es editorial grande (búsqueda por años)

    Output:
    * Archivo .csv con las fichas técnicas de los libros
    """
    print(f"\n{'='*60}")
    print(f"Editorial: {nombre_editorial}")
    print(f"{'='*60}")

    # Fase 1: recoger listado de libros
    print("Leyendo catálogo...")
    if es_grande:
        libros = buscar_editorial_grande(id_editorial, nombre_editorial, inicio, fin)
    else:
        libros = buscar_editorial(id_editorial, nombre_editorial, inicio, fin)

    print(f"✓ {len(libros)} libros encontrados.")

    if not libros:
        print("No hay libros que procesar.")
        return

    # Fase 2: extraer fichas técnicas
    print("Extrayendo fichas técnicas...")
    lista_libros = []

    for i, libro in enumerate(libros):
        print(f"  [{i+1}/{len(libros)}] {libro['titulo'][:50]}...")

        ficha = extraer_datos(libro["url"] + "#fichaTecnica")

        # Añadir campo traducción si no existe
        if "Traducción" not in ficha:
            ficha["Traducción"] = "Sin traducción"

        # Añadir datos del catálogo a la ficha
        ficha["Título"] = libro["titulo"]
        ficha["Precio"] = libro["precio"]
        ficha["URL"] = libro["url"]

        lista_libros.append(ficha)

    # Fase 3: guardar JSON
    os.makedirs("data", exist_ok=True)
    ruta_json = f"data/catalogos/catalogo_{nombre_editorial.lower()}.json"
    campos = lista_libros[0].keys()

    # Revisión del catálogo
    if os.path.exists(ruta_json):
        with open(ruta_json, "r", encoding="utf-8") as f:
            libros_existentes = json.load(f)
    else:
        libros_existentes = []

    # Añadir nuevos libros al final
    libros_existentes.extend(lista_libros)

    # Guardar el resultado completo
    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(libros_existentes, f, ensure_ascii=False, indent=2)

    print(f"✓ {len(lista_libros)} fichas guardadas en {ruta_json}.")


# ======================================================================================
# CONTROL DE FLUJO INTERACTIVO
# ======================================================================================

def control_flujo(ruta_editoriales="data/json/editoriales.json", ruta_estado="data/json/estado.json"):
    """
    Función principal que gestiona el flujo interactivo del scraping.

    Lee el fichero de editoriales (editoriales.json) con la información de cada una,
    consulta el estado guardado (estado.json) para saber cuáles ya están procesadas,
    y pregunta al usuario qué hacer con cada una.

    Estructura esperada de editoriales.json:
    {
        "debolsillo": {
            "id": "debolsillo_179709",
            "grande": false,
            "intervalo_max": [1, 200]   <- páginas si normal, años si grande
        },
        "espasa": {
            "id": "espasa_76490",
            "grande": true,
            "intervalo_max": [1990, 2024]
        }
    }

    Estructura de estado.json (se genera automáticamente):
    {
        "debolsillo": {
            "ultimo": 200    <- última página/año procesada
        }
    }
    """
    global driver

    # Cargar información de editoriales
    if not os.path.exists(ruta_editoriales):
        print(f"Error: no se encuentra '{ruta_editoriales}'.")
        return

    with open(ruta_editoriales, "r", encoding="utf-8") as f:
        info_editoriales = json.load(f)

    # Cargar estado previo si existe, o empezar desde cero
    if os.path.exists(ruta_estado):
        with open(ruta_estado, "r", encoding="utf-8") as f:
            estado = json.load(f)
    else:
        estado = {}

    # Iniciar el driver una sola vez para toda la sesión
    print("Iniciando navegador...")
    driver = crear_driver()
    print("✓ Navegador listo.\n")

    try:
        for nombre, info in info_editoriales.items():
            maximo = info["intervalo"][1]
            es_grande = info["grande"]
            id_editorial = info["id"]

            # Comprobar si ya está completamente procesada
            if nombre in estado and estado[nombre]["ultimo"] >= maximo:
                print(f"{nombre}: ya procesada completamente, saltando...")
                continue

            # Determinar punto de inicio (desde el principio o desde donde se dejó)
            if nombre in estado:
                ultimo_guardado = estado[nombre]["ultimo"]
                inicio_sugerido = ultimo_guardado+1 # retomamos desde el último guardado
                msg = f"{nombre}: proceso iniciado. Último guardado en {ultimo_guardado}. ¿Continuar? [Y/N]: "
            else:
                inicio_sugerido = info["intervalo"][0]
                msg = f"{nombre}: aún no procesada. ¿Comenzar? [Y/N]: "

            respuesta = input(msg).strip().upper()

            if respuesta == "N":
                print(f"Saltando {nombre}.\n")
                continue

            elif respuesta == "Y":
                # Pedir intervalo al usuario
                tipo = "año" if es_grande else "página"
                print(f"Inicio sugerido: {inicio_sugerido} | Máximo disponible: {maximo}")

                entrada_inicio = input(f"Introduce {tipo} de inicio [{inicio_sugerido}]: ").strip()
                entrada_fin = input(f"Introduce {tipo} de fin (máx. {maximo}): ").strip()

                # Usar valores sugeridos si el usuario no introduce nada
                inicio = int(entrada_inicio) if entrada_inicio else inicio_sugerido
                fin = min(int(entrada_fin), maximo)  # nunca superar el máximo

                # Ejecutar el scraping
                scrapear_editorial(id_editorial, nombre, inicio, fin, es_grande)

                # Actualizar y guardar estado
                if nombre not in estado:
                    estado[nombre] = {}
                estado[nombre]["ultimo"] = fin

                with open(ruta_estado, "w", encoding="utf-8") as f:
                    json.dump(estado, f, ensure_ascii=False, indent=2)

                print(f"Estado guardado: {nombre} → hasta {tipo} {fin}.\n")

            else:
                print("Respuesta no válida, saltando.\n")

    finally:
        # Cerrar el navegador siempre, aunque haya errores
        driver.quit()
        print("\nNavegador cerrado.")


# ======================================================================================
# PUNTO DE ENTRADA
# ======================================================================================

if __name__ == "__main__":
    control_flujo()

### 2. Creación del DataFrame base para guardar en parquet

In [1]:
# ======================================================================================
# CREACIÓN DEL DATAFRAME BASE
# ======================================================================================

import json
import pandas as pd
import numpy as np
from pathlib import Path


def crear_df(ruta_catalogos="data/prueba"):
    path = Path(ruta_catalogos)
    df = pd.DataFrame({})
    jsons = []

    print("="*50,"\nCreando DataFrame con todos los libros\n","="*50)
    for archivo in path.iterdir():
        if archivo.is_file():
            print(f"Añadiendo {archivo.name}")
            editorial = pd.read_json(archivo.absolute())
            jsons.append(editorial)

    df = pd.concat(jsons, axis=0)

    # Borrar filas repetidas
    print("Catálogos convertidos a DataFrame. Eliminando filas duplicadas...")
    df.drop_duplicates(subset=['EAN'], keep='first', inplace=True)

    # Limpiado de nombre de columnas
    df.columns = df.columns.str.strip().str.lower().str.translate(str.maketrans({"á": "a", "é": "e", "í": "i", "ó":"o", "ú": "u", "º": "", " ": "_"}))

    # Transformando columnas con tipos mixtos

    
    return df
    

    print("DataFrame creado con éxito.")

# def limpiar_df está abajo

def crear_parquet(df: pd.DataFrame, ruta_guardado="data/parquet/lista_libros.parquet"):

    # Creando .parquet
    df.to_parquet(ruta_guardado, index=False)
    print("Archivo .parquet creado con éxito.")

In [2]:
# varios fallos
# columnas con valores mixtos (str y list)
# hay que añadir una columna para las urls de las imágenes: https://static.cegal.es/imagenes/marcadas/9788466/978846639275.gif
data_raw = crear_df()

Creando DataFrame con todos los libros
Añadiendo catalogo_acantilado.json
Añadiendo catalogo_alfaguara.json
Añadiendo catalogo_alianza editorial.json
Añadiendo catalogo_anagrama.json
Añadiendo catalogo_booket.json
Añadiendo catalogo_debolsillo.json
Añadiendo catalogo_destino.json
Añadiendo catalogo_ediciones akal.json
Añadiendo catalogo_editorial anagrama.json
Añadiendo catalogo_espasa.json
Añadiendo catalogo_maxi tusquets.json
Añadiendo catalogo_planeta.json
Añadiendo catalogo_plaza & janes.json
Añadiendo catalogo_rae.json
Añadiendo catalogo_seix barral.json
Añadiendo catalogo_siruela.json
Añadiendo catalogo_tusquets editores.json
Añadiendo catalogo_tusquets.json
Catálogos convertidos a DataFrame. Eliminando filas duplicadas...


In [7]:
def primer_valor_valido(x):
    valores = x.dropna()
    return x.fillna(valores.iloc[0] if not valores.empty else np.nan)

def primer_valor_valido_lista(x):
    valores = x.dropna()
    if valores.empty:
        return x
    primer_valor = valores.iloc[0]
    return x.apply(lambda v: primer_valor if v is None or (isinstance(v, float) and np.isnan(v)) else v)


def limpiar_df(df: pd.DataFrame, cols_lista=['categorias', 'autoria', 'traductor'], cols_borrar=['isbn', 'editorial'], dict_ed={}):

    """ 
    Limpia el DataFrame del catálogo completo. 
    1. Elimina libros con título, autor o ean faltante (datos imprescindibles)
    2. Borra filas repetidas
    3. Deja libros sólo en español
    4. Convierte columnas con tipos mixtos para poder guardarlas en parquet
    5. Parsea fechas
    6. Extrae valores numéricos para dimensiones, peso y precio
    7. Rellena filas vacías de sinopsis, dimensiones, peso, precio y encuadernación

    Parámetros:
    * **df**: DataFrame (referiblemente el resultado de `crear_df()')
    * **cols_lista**: lista de columnas cuyo tipo se debe convertir a lista
    * **cols_borrar**: lista de columnas a borrar
    * **dict_ed**: diccionario con los ids de cada editorial para incluirlos en el DataFrame

    Output:
    * DataFrame limpio
    """

    df_clean = df.copy()

    # Eliminar libros con datos importantes faltantes
    df_clean.dropna(subset=['titulo', 'autoria', 'ean'], inplace=True)

    # Borrar filas repetidas
    df_clean.drop_duplicates(subset=['ean'], keep='first', inplace=True)

    # Dejar libros solo en español
    df_clean = df_clean[df_clean['idioma_de_publicacion'].str.strip().str.lower() == 'castellano']

    # Cambiando columnas con tipos mixtos
    for col in cols_lista:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].apply(lambda x: x if isinstance(x, list) else ([x] if pd.notna(x) else []))

    # Parsear fecha de publicación
    df_clean['fecha_publicacion'] = pd.to_datetime(df_clean['fecha_publicacion'], format="%d-%m-%Y", errors='coerce')

    # Cambio formato de dimensiones (ancho x alto mm. -> [ancho, alto])
    df_clean['dimensiones'] = df_clean['dimensiones'].apply(
        lambda x: [float(n.strip()) for n in x.replace('mm.', '').replace('mm', '').split('x') if n.strip()] if pd.notna(x) else np.nan
    )

    # Cambio formato para peso
    df_clean['peso'] = df_clean['peso'].apply(
        lambda x: float(x.replace(' gramos', '').strip()) if pd.notna(x) else np.nan
    )

    # Cambio de tipo de precio
    df_clean['precio'] = df_clean['precio'].apply(
        lambda x: float(x.replace(' €', '').replace('.', '').replace(',', '.').strip()) if pd.notna(x) and x != 'Más información' else np.nan
    )

    # hay que cambiar esto para que (probar también sin coleccion)
    # dimensiones dejar así
    # peso según dimensiones también (producto de las dimensiones) y el nº páginas/grueso si no nº páginas
    # añadir grueso según dimensión y nº páginas (si no hay nº páginas por peso)
    # alternativa más simple para primer_valor_válido_lista
    df_clean['dimensiones'] = df_clean.groupby(['editorial', 'coleccion'])['dimensiones'].transform(primer_valor_valido_lista)
    df_clean['peso'] = df_clean.groupby(['editorial', 'coleccion'])['peso'].transform('median')
    df_clean['precio'] = df_clean.groupby(['editorial', 'coleccion'])['precio'].transform('median')


    # Relleno de sinopsis vacías
    df_clean['sinopsis'] = df_clean['sinopsis'].fillna('Sin sinopsis')

    # Columna id editoriales
    if dict_ed:
        df_clean['id_editorial'] = df_clean['editorial'].map(dict_ed)

    # Borrar columnas innecesarias
    df_clean.drop(columns=cols_borrar, inplace=True, errors='ignore')

    return df_clean

id_dicts = {
    'PLAZA & JANES': 'id_1',
    'RAE': 'id_2',
    'Seix Barral': 'id_3'
}
data = limpiar_df(data_raw, dict_ed=id_dicts)
print(data_raw.isna().sum())
print(data.isna().sum())

isbn                                  0
ean                                   0
editorial                             0
coleccion                          8245
fecha_publicacion                   328
autoria                            2622
traduccion                            0
n_paginas                          5100
encuadernacion                     6142
dimensiones                       16003
grueso                            44326
peso                              28194
categorias                         5213
pais_de_publicacion                5981
idioma_de_publicacion                14
idioma_original                   20960
sinopsis                              0
titulo                                0
precio                                0
url                                   0
edicion_literaria                 77583
lectura                           77600
epilogo                           77582
ilustracion                       75924
prologo                           77257


In [26]:
dim = [x for x in data['dimensiones'].dropna() if len(x)==2]
lista = []
for x in dim:
    lista.append(x[0]*x[1])


print(len(set(lista)))

115


### 3. Merge de los datos del SPI

In [ ]:
# ======================================================================================
# DATOS DE SPI
# ======================================================================================

from functools import reduce

def merge_spi(lista_editoriales, ruta_ed="data/json/editoriales.json", ruta_spi="data/bronze/spi"):
    
    with open(ruta_ed, "r", encoding="utf-8") as f:
        editoriales = json.load(f)

    clasificaciones = list(Path(ruta_spi).iterdir())
    dfs = []
    for ruta_cla in clasificaciones:
        print(f"\nIncluyendo {ruta_cla.name}...")
        with open(ruta_cla, "r", encoding="utf-8") as f:
            dfs.append(pd.DataFrame(json.load(f)))

    df = reduce(lambda left, right: left.merge(right, on="Editorial", how='outer'), dfs)

    nombre_spi = {editoriales[ed]["nombre_spi"]: ed for ed in lista_editoriales}
    id_dict = {editoriales[ed]["nombre_spi"]: editoriales[ed]["id"] for ed in lista_editoriales}

    df_selection = df[df['Editorial'].isin(nombre_spi.keys())].copy()
    df_selection['Editorial'] = df_selection["Editorial"].map(nombre_spi)
    df_selection['id'] = df_selection["Editorial"].map(id_dict)

    return df_selection

Fin de la capa **bronze**